In [1]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "customerretentionintelligence"
DATASET_ID = "retention_ml"

client = bigquery.Client(project=PROJECT_ID)

print("Cliente conectado al proyecto:", client.project)

Cliente conectado al proyecto: customerretentionintelligence


In [2]:
query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.raw_orders`
LIMIT 10
"""

df = client.query(query).to_dataframe()

df.head()

,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,1408,1146,Cancelled,F,2023-11-05 11:38:35+00:00,NaT,NaT,NaT,2
1,1735,1417,Cancelled,F,2026-07-23 01:31:19+00:00,NaT,NaT,NaT,1
2,1890,1540,Cancelled,F,2023-06-26 13:30:14+00:00,NaT,NaT,NaT,2
3,3363,2710,Cancelled,F,2024-10-25 12:03:49+00:00,NaT,NaT,NaT,2
4,3543,2846,Cancelled,F,2021-09-07 02:16:56+00:00,NaT,NaT,NaT,1


In [3]:
query_clean_users = f"""
SELECT
    user_id,
    age,
    gender,
    country,
    traffic_source,
    created_at
FROM `{PROJECT_ID}.{DATASET_ID}.clean_users`
"""

job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=1_073_741_824
)

df_users = client.query(
    query_clean_users,
    job_config=job_config
).to_dataframe()

print("Dimensiones:", df_users.shape)
df_users.head()

Dimensiones: (100000, 6)


,user_id,age,gender,country,traffic_source,created_at
0,43768,12,F,Japan,Organic,2026-09-04 06:27:23.609832+00:00
1,18983,12,F,Japan,Search,2019-04-15 18:51:00+00:00
2,19825,12,M,Japan,Search,2025-07-24 09:37:00+00:00
3,66367,12,M,Japan,Search,2019-03-19 18:53:00+00:00
4,13600,12,M,United States,Search,2026-06-16 16:17:00+00:00


In [4]:
query_order_summary = f"""
SELECT
    user_id,
    COUNT(DISTINCT order_id) AS n_orders,
    MIN(DATE(created_at)) AS first_order_date,
    MAX(DATE(created_at)) AS last_order_date
FROM `{PROJECT_ID}.{DATASET_ID}.clean_orders`
GROUP BY user_id
ORDER BY n_orders DESC
"""

df_order_summary = client.query(
    query_order_summary,
    job_config=job_config
).to_dataframe()

print("Dimensiones:", df_order_summary.shape)
df_order_summary.head(10)

Dimensiones: (79684, 4)


,user_id,n_orders,first_order_date,last_order_date
0,55163,4,2022-08-16,2024-08-31
1,56591,4,2026-05-21,2026-08-18
2,27784,4,2024-08-01,2025-10-10
3,23014,4,2023-11-30,2026-04-12
4,46714,4,2023-02-14,2026-04-17
5,28549,4,2026-09-03,2026-09-05
6,80227,4,2024-08-30,2026-02-22
7,6452,4,2024-08-13,2026-06-07
8,10091,4,2025-01-06,2026-04-27
9,43995,4,2020-06-04,2026-04-06
